In [0]:
--Finaliser la couche Gold :
--Appropriez-vous le code de la création
--des tables de la couches gold ainsi que
--les notebooks de chargement.
--Identifiez des pistes d’améliorations.

-- Exemple de version améliorée pour la dimension

MERGE INTO gold.dim_geography AS tgt
USING (
  SELECT
    TRY_CAST(address_id AS INT) AS geo_address_id,
    COALESCE(NULLIF(TRIM(address_line1), ''), 'N/A') AS geo_address_line_1,
    COALESCE(NULLIF(TRIM(address_line2), ''), 'N/A') AS geo_address_line_2,
    COALESCE(NULLIF(TRIM(city), ''), 'N/A')          AS geo_city,
    COALESCE(NULLIF(TRIM(state_province), ''), 'N/A') AS geo_state_province,
    COALESCE(NULLIF(TRIM(country_region), ''), 'N/A') AS geo_country_region,
    COALESCE(NULLIF(TRIM(postal_code), ''), 'N/A')   AS geo_postal_code
  FROM silver.address
  WHERE _tf_valid_to IS NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY address_id
    ORDER BY modified_date DESC
  ) = 1
) AS src
ON tgt.geo_address_id = src.geo_address_id

WHEN MATCHED AND (
       tgt.geo_address_line_1  IS DISTINCT FROM src.geo_address_line_1
    OR tgt.geo_address_line_2  IS DISTINCT FROM src.geo_address_line_2
    OR tgt.geo_city            IS DISTINCT FROM src.geo_city
    OR tgt.geo_state_province  IS DISTINCT FROM src.geo_state_province
    OR tgt.geo_country_region  IS DISTINCT FROM src.geo_country_region
    OR tgt.geo_postal_code     IS DISTINCT FROM src.geo_postal_code
) THEN
  UPDATE SET
    tgt.geo_address_line_1 = src.geo_address_line_1,
    tgt.geo_address_line_2 = src.geo_address_line_2,
    tgt.geo_city = src.geo_city,
    tgt.geo_state_province = src.geo_state_province,
    tgt.geo_country_region = src.geo_country_region,
    tgt.geo_postal_code = src.geo_postal_code,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED AND src.geo_address_id IS NOT NULL THEN
  INSERT (
    geo_address_id,
    geo_address_line_1,
    geo_address_line_2,
    geo_city,
    geo_state_province,
    geo_country_region,
    geo_postal_code,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.geo_address_id,
    src.geo_address_line_1,
    src.geo_address_line_2,
    src.geo_city,
    src.geo_state_province,
    src.geo_country_region,
    src.geo_postal_code,
    load_date,
    load_date
  )
;

-- Exemple de version améliorée pour la table de fait

CREATE OR REPLACE TEMP VIEW _tmp_fact_sales AS
SELECT *
FROM (
  SELECT
    CAST(soh.sales_order_id AS INT)        AS sales_order_id,
    CAST(sod.sales_order_detail_id AS INT) AS sales_order_detail_id,

    CAST(date_format(soh.order_date, 'yyyyMMdd') AS INT) AS _tf_dim_calendar_id,
    COALESCE(cust._tf_dim_customer_id, -9)        AS _tf_dim_customer_id,
    COALESCE(geo._tf_dim_geography_id, -9)        AS _tf_dim_geography_id,

    COALESCE(CAST(sod.order_qty AS SMALLINT), 0)                AS sales_order_qty,
    COALESCE(CAST(sod.unit_price AS DECIMAL(19,4)), 0)          AS sales_unit_price,
    COALESCE(CAST(sod.unit_price_discount AS DECIMAL(19,4)), 0) AS sales_unit_price_discount,
    COALESCE(CAST(sod.line_total AS DECIMAL(38,6)), 0)           AS sales_line_total,

    ROW_NUMBER() OVER (
      PARTITION BY soh.sales_order_id, sod.sales_order_detail_id
      ORDER BY soh.modified_date DESC
    ) AS rn

  FROM silver.sales_order_detail sod
  INNER JOIN silver.sales_order_header soh 
    ON sod.sales_order_id = soh.sales_order_id 
   AND soh._tf_valid_to IS NULL

  LEFT JOIN silver.customer c 
    ON soh.customer_id = c.customer_id 
   AND c._tf_valid_to IS NULL

  LEFT JOIN gold.dim_customer cust
    ON c.customer_id = cust.cust_customer_id

  LEFT JOIN silver.address a 
    ON soh.bill_to_address_id = a.address_id 
   AND a._tf_valid_to IS NULL

  LEFT JOIN gold.dim_geography geo 
    ON a.address_id = geo.geo_address_id

  WHERE sod._tf_valid_to IS NULL
)
WHERE rn = 1;

MERGE INTO gold.fact_sales AS tgt
USING _tmp_fact_sales AS src
ON tgt.sales_order_id = src.sales_order_id
AND tgt.sales_order_detail_id = src.sales_order_detail_id

WHEN MATCHED AND (
       tgt._tf_dim_calendar_id      IS DISTINCT FROM src._tf_dim_calendar_id
    OR tgt._tf_dim_customer_id      IS DISTINCT FROM src._tf_dim_customer_id
    OR tgt._tf_dim_geography_id     IS DISTINCT FROM src._tf_dim_geography_id
    OR tgt.sales_order_qty          IS DISTINCT FROM src.sales_order_qty
    OR tgt.sales_unit_price         IS DISTINCT FROM src.sales_unit_price
    OR tgt.sales_unit_price_discount IS DISTINCT FROM src.sales_unit_price_discount
    OR tgt.sales_line_total         IS DISTINCT FROM src.sales_line_total
) THEN
  UPDATE SET
    tgt._tf_dim_calendar_id       = src._tf_dim_calendar_id,
    tgt._tf_dim_customer_id       = src._tf_dim_customer_id,
    tgt._tf_dim_geography_id      = src._tf_dim_geography_id,
    tgt.sales_order_qty           = src.sales_order_qty,
    tgt.sales_unit_price          = src.sales_unit_price,
    tgt.sales_unit_price_discount = src.sales_unit_price_discount,
    tgt.sales_line_total          = src.sales_line_total,
    tgt._tf_update_date           = load_date

WHEN NOT MATCHED THEN
  INSERT (
    sales_order_id,
    sales_order_detail_id,
    _tf_dim_calendar_id,
    _tf_dim_customer_id,
    _tf_dim_geography_id,
    sales_order_qty,
    sales_unit_price,
    sales_unit_price_discount,
    sales_line_total,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.sales_order_id,
    src.sales_order_detail_id,
    src._tf_dim_calendar_id,
    src._tf_dim_customer_id,
    src._tf_dim_geography_id,
    src.sales_order_qty,
    src.sales_unit_price,
    src.sales_unit_price_discount,
    src.sales_line_total,
    load_date,
    load_date
  );